# ⚡ FreightQuote AI — Milestone 2
### Enterprise Multi-Agent Logistics Intelligence Platform
This notebook trains 3 independent ML Agents (Dynamic Pricing R² ≥ 0.90, Route Delay ROC-AUC, Carrier Compliance ROC-AUC), logs model metadata to SQLite `ml_models`, initializes `Qwen2.5-3B-Instruct` 4-bit LLM Copilot, and launches Streamlit via Ngrok.

## Step 1 — Install Dependencies

In [ ]:
!pip install -q streamlit streamlit-option-menu pyjwt bcrypt plotly pandas numpy scikit-learn joblib transformers accelerate bitsandbytes faker kaggle pyngrok python-dotenv

## Step 2 — Configure Secrets & Environment

In [ ]:
import os

def _get_secret(key):
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
HF_TOKEN        = _get_secret("HF_TOKEN")
EMAIL_ADDRESS   = _get_secret("EMAIL_ADDRESS")
EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
KAGGLE_USERNAME = _get_secret("KAGGLE_USERNAME")
KAGGLE_KEY      = _get_secret("KAGGLE_KEY")

print("🔑 NGROK_AUTHTOKEN Configured:", "✅ Yes" if NGROK_AUTHTOKEN else "⚠️ No")
print("🔑 HF_TOKEN Configured:       ", "✅ Yes" if HF_TOKEN else "⚠️ No")
print("🔑 KAGGLE Credentials:        ", "✅ Yes" if (KAGGLE_USERNAME and KAGGLE_KEY) else "⚠️ Synthetic Fallback")


## Step 3 — Train 3 Independent ML Champion Models

In [ ]:
import train_ml_freight, db

# Run Master ML Pipeline
results = train_ml_freight.train_all_agents()

# Display logged SQLite metadata
models = db.get_all_ml_models_metadata()
print(f"\n🏆 Champion Models Logged in SQLite ml_models Table ({len(models)} Total):")
for m in models:
    print(f" - {m['agent_name']}: {m['algorithm_name']} | Metrics: {m['metrics']} | Saved to: {m['model_file_path']}")


## Step 4 — Initialize Qwen2.5-3B-Instruct 4-Bit LLM Copilot Engine

In [ ]:
import llm_engine_freight

engine = llm_engine_freight.LogisticsLLMEngine()

# Test Multi-Agent Structured Audit Output
audit_output = engine.produce_structured_audit({
    "origin_port": "Nhava Sheva (INNSA)",
    "dest_port": "Rotterdam (NLRTM)",
    "distance_miles": 6800,
    "cargo_weight_tons": 22.0,
    "container_type_label": "40ft High Cube Dry"
})

print("✅ Multi-Agent Audit Output JSON:")
import json
print(json.dumps(audit_output, indent=2))


## Step 5 — Launch Streamlit Server with Ngrok Proxy

In [ ]:
import subprocess
from pyngrok import ngrok

ngrok.kill()
if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    public_url = ngrok.connect(8501).public_url
    print(f"🚀 PUBLIC NGROK URL: {public_url}")
else:
    print("⚠️ Running locally on port 8501.")

process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
print("✅ Streamlit Server launched!")
